# Landing Page V2 A/B Test Walkthrough

This notebook frames the project as a product analytics case study. The business question is simple: **should Landing Page V2 ship to all traffic?**

The recommendation depends on the primary conversion result and guardrail metrics for bounce, revenue per visitor, and sample size balance.

## 1. Experiment Setup

- Control: current landing page
- Treatment: Landing Page V2
- Primary metric: conversion rate
- Guardrails: bounce rate, revenue per visitor, sample size balance
- Decision rule: ship only if conversion improves significantly and guardrails do not show blocking risk

In [ ]:
from pathlib import Path
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.engine.simulator import ExperimentSimulator
from src.engine.stats import ExperimentStats

with open(PROJECT_ROOT / "config/experiment_config.yaml", "r") as file:
    config = yaml.safe_load(file)

params = config["parameters"]
experiment_id = config["experiment_name"]
alpha = params["alpha"]

required_n = ExperimentStats.calculate_sample_size(
    params["baseline_value"],
    params["mde"],
    alpha,
    params["power"],
)

print(f"Experiment: {experiment_id}")
print(f"Configured users: {params['n_users']:,}")
print(f"Required sample size per variant for target MDE: {required_n:,}")

## 2. Simulate Experiment Data

The dataset mirrors the fields a product analyst would expect after an experiment export: assignment, conversion, bounce, and revenue/order value proxies.

In [ ]:
df = ExperimentSimulator.generate_data(
    n_users=params["n_users"],
    exp_id=experiment_id,
    true_p_a=params["baseline_value"],
    true_p_b=params["baseline_value"] * (1 + params["mde"]),
    seed=params["seed"],
)

df.head()

In [ ]:
df.groupby("variant").agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean"),
    bounce_rate=("bounced", "mean"),
    revenue_per_visitor=("revenue", "mean"),
)

## 3. Primary Metric and Statistical Test

The primary metric is conversion rate. The test compares treatment and control using a two-sample z-test for proportions.

In [ ]:
primary = ExperimentStats.analyze_binary_metric(df, "converted", alpha=alpha)
primary

## 4. Guardrail Metrics

Guardrails protect the business from shipping a change that improves conversion while damaging the user experience or monetization quality.

In [ ]:
bounce = ExperimentStats.analyze_binary_metric(df, "bounced", alpha=alpha)
bounce["name"] = "Bounce rate"
bounce["violated"] = bool(bounce["significant"] and bounce["lift"] > 0.02)

revenue = ExperimentStats.analyze_mean_metric(df, "revenue", alpha=alpha)
revenue["name"] = "Revenue per visitor"
revenue["violated"] = bool(revenue["significant"] and revenue["lift"] < -0.02)

balance = ExperimentStats.analyze_sample_balance(df)
balance["name"] = "Sample size balance"

guardrails = [bounce, revenue, balance]

pd.DataFrame([
    {
        "metric": g["name"],
        "lift": g.get("lift"),
        "p_value": g.get("p_value"),
        "violated": g.get("violated", False),
    }
    for g in guardrails
])

## 5. Decision Recommendation

The treatment should ship only when the primary metric is positive and statistically significant, with no blocking guardrail risk.

In [ ]:
decision = ExperimentStats.get_decision(primary, guardrails)
print(decision)

print(f"Control conversion: {primary['p_a']:.2%}")
print(f"Treatment conversion: {primary['p_b']:.2%}")
print(f"Relative lift: {primary['lift']:.2%}")
print(f"p-value: {primary['p_value']}")
print(f"95% CI for absolute conversion change: {primary['ci'][0]:.2%} to {primary['ci'][1]:.2%}")

## 6. Visual Summary

The chart compares variants, shows uncertainty around the treatment effect, and summarizes guardrail movement.

In [ ]:
chart_path = PROJECT_ROOT / "data/simulated/experiment_results_chart.png"
chart_path.parent.mkdir(parents=True, exist_ok=True)
ExperimentStats.visualize_results(primary, str(chart_path), guardrails=guardrails)
chart_path